In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


In [2]:
import numpy as np
import pandas as pd
import warnings
 
warnings.filterwarnings("ignore")
 
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e7"
 
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
 
print(train.shape, test.shape)
train.head()

(690088, 15) (295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [3]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import optuna
 
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [4]:
TARGET = "health_condition"
 
le = LabelEncoder()
y_all = le.fit_transform(train[TARGET])
classes = le.classes_
n_classes = len(classes)
print("Classes (in label-encoded order):", list(classes))
 
# Class priors, in label-encoded order (0, 1, 2, ...)
class_counts = np.bincount(y_all)
class_priors = class_counts / class_counts.sum()
print("Class priors:", dict(zip(classes, class_priors)))
 
# Balanced sample weights: inverse of prior, normalized
class_weight_dict = {
    c: class_counts.sum() / (n_classes * class_counts[c]) for c in range(n_classes)
}
print("Class weights:", {classes[k]: v for k, v in class_weight_dict.items()})
 
sample_weight_all = np.array([class_weight_dict[v] for v in y_all])
 
X_all_raw = train.drop(columns=["id", TARGET])
X_test_raw = test.drop(columns=["id"])

Classes (in label-encoded order): ['at-risk', 'fit', 'unhealthy']
Class priors: {'at-risk': np.float64(0.8586745458550213), 'fit': np.float64(0.057678151192311705), 'unhealthy': np.float64(0.08364730295266691)}
Class weights: {'at-risk': np.float64(0.38819519565636845), 'fit': np.float64(5.779195873007898), 'unhealthy': np.float64(3.9849860254544613)}


In [5]:
def add_features(df):
    df = df.copy()
 
    df["activity_per_min"] = df["step_count"] / (df["exercise_duration"] + 1)
    df["calories_per_step"] = df["calorie_expenditure"] / (df["step_count"] + 1)
    df["water_per_bmi"] = df["water_intake"] / (df["bmi"] + 1)
    df["sleep_bmi"] = df["sleep_duration"] * df["bmi"]
    df["heart_bmi"] = df["heart_rate"] * df["bmi"]
    df["bmi_squared"] = df["bmi"] ** 2
    df["heart_per_sleep"] = df["heart_rate"] / (df["sleep_duration"] + 1)
    df["exercise_per_bmi"] = df["exercise_duration"] / (df["bmi"] + 1)
    df["water_per_exercise"] = df["water_intake"] / (df["exercise_duration"] + 1)
    df["calories_per_min"] = df["calorie_expenditure"] / (df["exercise_duration"] + 1)
 
    # Additional composite / bucketed features
    df["cal_per_stepcount"] = df["calorie_expenditure"] / (
        df["step_count"] + df["exercise_duration"] + 1
    )
    df["sleep_efficiency"] = df["sleep_duration"] / (df["heart_rate"] + 1)
    df["water_sleep"] = df["water_intake"] * df["sleep_duration"]
    df["overall_load"] = (
        df["step_count"] * df["exercise_duration"] / (df["sleep_duration"] + 1)
    )
    df["bmi_bucket"] = pd.cut(
        df["bmi"], bins=[0, 18.5, 25, 30, 35, 100], labels=False
    )
    df["hr_bucket"] = pd.cut(
        df["heart_rate"], bins=[0, 60, 80, 100, 250], labels=False
    )
 
    return df
 
 
def preprocess(train_df, valid_df=None):
    train_df = train_df.copy()
    if valid_df is not None:
        valid_df = valid_df.copy()
 
    numeric_cols = train_df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        median = train_df[col].median()
        train_df[col] = train_df[col].fillna(median)
        if valid_df is not None:
            valid_df[col] = valid_df[col].fillna(median)
 
    cat_cols = train_df.select_dtypes(include="object").columns
    for col in cat_cols:
        train_df[col] = train_df[col].fillna("Missing").astype("category")
        if valid_df is not None:
            valid_df[col] = valid_df[col].fillna("Missing").astype("category")
 
    if valid_df is None:
        return train_df
    return train_df, valid_df
 
 
X_all = add_features(X_all_raw)
X_test_all = add_features(X_test_raw)
 
categorical_features = X_all.select_dtypes(include="object").columns.tolist()
print("Num features:", X_all.shape[1])
print("Categorical features:", categorical_features)

Num features: 29
Categorical features: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [6]:
def prior_adjusted_predict(proba, priors=class_priors):
    """
    Bayes-optimal-ish rule for balanced accuracy: divide each class's
    predicted probability by its training prior before taking argmax.
    This is equivalent to re-weighting toward a uniform prior at
    decision time, which is what balanced accuracy implicitly assumes.
    """
    adjusted = proba / priors
    return adjusted.argmax(axis=1)


In [7]:
SEARCH_FOLDS = 2
SEARCH_TRIALS = 2  # bump up if you have time budget
 
 
def objective(trial):
    params = {
        "objective": "multiclass",
        "n_estimators": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 96),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }
 
    skf = StratifiedKFold(n_splits=SEARCH_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
 
    for tr_idx, va_idx in skf.split(X_all, y_all):
        X_tr, X_va = X_all.iloc[tr_idx].copy(), X_all.iloc[va_idx].copy()
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]
        w_tr = sample_weight_all[tr_idx]
 
        X_tr, X_va = preprocess(X_tr, X_va)
        for col in categorical_features:
            X_va[col] = pd.Categorical(
                X_va[col], categories=X_tr[col].cat.categories
            )
 
        model = LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            categorical_feature=categorical_features,
            eval_set=[(X_va, y_va)],
            eval_metric="multi_logloss",
            callbacks=[early_stopping(100, verbose=False), log_evaluation(0)],
        )
 
        proba = model.predict_proba(X_va)
        preds = prior_adjusted_predict(proba)
        fold_scores.append(balanced_accuracy_score(y_va, preds))
 
    return float(np.mean(fold_scores))
 
 
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=SEARCH_TRIALS, show_progress_bar=True)
 
print("Best OOF balanced accuracy (search CV):", study.best_value)
print("Best params:", study.best_params)
 

  0%|          | 0/2 [00:00<?, ?it/s]

Best OOF balanced accuracy (search CV): 0.9466083821040263
Best params: {'learning_rate': 0.01782581322351515, 'num_leaves': 88, 'max_depth': 12, 'min_child_samples': 90, 'subsample': 0.6762541224049752, 'colsample_bytree': 0.8672844062715556, 'reg_alpha': 3.6828077181487693, 'reg_lambda': 1.4275189032003128}


In [8]:
best_params = {
    "objective": "multiclass",
    "n_estimators": 5000,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
    **study.best_params,
}
 
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
oof_proba = np.zeros((len(X_all), n_classes))
test_proba = np.zeros((len(X_test_all), n_classes))
models = []
 
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all), start=1):
    print("=" * 60, f"\nFold {fold}")
 
    X_tr, X_va = X_all.iloc[tr_idx].copy(), X_all.iloc[va_idx].copy()
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]
    w_tr = sample_weight_all[tr_idx]
 
    X_tr, X_va = preprocess(X_tr, X_va)
    X_te = X_test_all.copy()
    for col in categorical_features:
        X_va[col] = pd.Categorical(X_va[col], categories=X_tr[col].cat.categories)
        X_te[col] = pd.Categorical(X_te[col], categories=X_tr[col].cat.categories)
 
    model = LGBMClassifier(**best_params)
    model.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        categorical_feature=categorical_features,
        eval_set=[(X_va, y_va)],
        eval_metric="multi_logloss",
        callbacks=[early_stopping(150, verbose=False), log_evaluation(0)],
    )
 
    oof_proba[va_idx] = model.predict_proba(X_va)
    test_proba += model.predict_proba(X_te) / skf.n_splits
    models.append(model)
 
    fold_preds_raw = oof_proba[va_idx].argmax(axis=1)
    fold_preds_adj = prior_adjusted_predict(oof_proba[va_idx])
 
    print(
        f"Fold {fold} balanced accuracy "
        f"(raw argmax): {balanced_accuracy_score(y_va, fold_preds_raw):.6f}  "
        f"(prior-adjusted): {balanced_accuracy_score(y_va, fold_preds_adj):.6f}"
    )

Fold 1
Fold 1 balanced accuracy (raw argmax): 0.936067  (prior-adjusted): 0.947825
Fold 2
Fold 2 balanced accuracy (raw argmax): 0.937817  (prior-adjusted): 0.949202
Fold 3
Fold 3 balanced accuracy (raw argmax): 0.936303  (prior-adjusted): 0.946277
Fold 4
Fold 4 balanced accuracy (raw argmax): 0.934970  (prior-adjusted): 0.947227
Fold 5
Fold 5 balanced accuracy (raw argmax): 0.934848  (prior-adjusted): 0.945896


In [9]:
oof_preds_raw = oof_proba.argmax(axis=1)
oof_preds_adj = prior_adjusted_predict(oof_proba)
 
score_raw = balanced_accuracy_score(y_all, oof_preds_raw)
score_adj = balanced_accuracy_score(y_all, oof_preds_adj)
 
print(f"\nOverall OOF balanced accuracy (raw argmax):      {score_raw:.6f}")
print(f"Overall OOF balanced accuracy (prior-adjusted):  {score_adj:.6f}")
 
print("\nConfusion matrix (prior-adjusted), rows=true, cols=pred:")
print(pd.DataFrame(
    confusion_matrix(y_all, oof_preds_adj),
    index=classes, columns=classes,
))
 
# Sanity check: prior-adjusted should be >= raw. If not, something's
# off (e.g. class_priors mis-ordered) — investigate before submitting.
assert score_adj >= score_raw - 1e-6, "Prior-adjusted rule underperformed raw argmax — check class ordering."
 
 
def objective(trial):
    params = {
        "objective": "multiclass",
        "n_estimators": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 96),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }
 
    skf = StratifiedKFold(n_splits=SEARCH_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
 
    for tr_idx, va_idx in skf.split(X_all, y_all):
        X_tr, X_va = X_all.iloc[tr_idx].copy(), X_all.iloc[va_idx].copy()
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]
        w_tr = sample_weight_all[tr_idx]
 
        X_tr, X_va = preprocess(X_tr, X_va)
        for col in categorical_features:
            X_va[col] = pd.Categorical(
                X_va[col], categories=X_tr[col].cat.categories
            )
 
        model = LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            categorical_feature=categorical_features,
            eval_set=[(X_va, y_va)],
            eval_metric="multi_logloss",
            callbacks=[early_stopping(100, verbose=False), log_evaluation(0)],
        )
 
        proba = model.predict_proba(X_va)
        preds = prior_adjusted_predict(proba)
        fold_scores.append(balanced_accuracy_score(y_va, preds))
 
    return float(np.mean(fold_scores))
 
 
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=SEARCH_TRIALS, show_progress_bar=True)
 
print("Best OOF balanced accuracy (search CV):", study.best_value)
print("Best params:", study.best_params)


Overall OOF balanced accuracy (raw argmax):      0.936001
Overall OOF balanced accuracy (prior-adjusted):  0.947286

Confusion matrix (prior-adjusted), rows=true, cols=pred:
           at-risk    fit  unhealthy
at-risk     546352  17349      28860
fit           1711  37848        244
unhealthy     1541    251      55932


  0%|          | 0/2 [00:00<?, ?it/s]

Best OOF balanced accuracy (search CV): 0.9445764616453752
Best params: {'learning_rate': 0.05762265679033534, 'num_leaves': 20, 'max_depth': 6, 'min_child_samples': 22, 'subsample': 0.6098572588358857, 'colsample_bytree': 0.6879990870793948, 'reg_alpha': 0.578558929425497, 'reg_lambda': 0.03770130397718224}


In [10]:
best_alpha, best_alpha_score = 1.0, score_adj
 
for alpha in np.linspace(0.5, 1.5, 21):
    adjusted = oof_proba / (class_priors ** alpha)
    preds = adjusted.argmax(axis=1)
    s = balanced_accuracy_score(y_all, preds)
    if s > best_alpha_score:
        best_alpha_score, best_alpha = s, alpha
 
print(f"\nBest prior-power alpha: {best_alpha:.2f}  OOF balanced accuracy: {best_alpha_score:.6f}")
 
 
def final_predict(proba, priors=class_priors, alpha=best_alpha):
    return (proba / (priors ** alpha)).argmax(axis=1)
 


Best prior-power alpha: 0.70  OOF balanced accuracy: 0.948748


In [11]:
test_preds = final_predict(test_proba)
 
submission = pd.DataFrame({
    "id": test["id"],
    TARGET: le.inverse_transform(test_preds),
})
submission.to_csv("submission.csv", index=False)
print(submission[TARGET].value_counts(normalize=True))
submission.head()
 

health_condition
at-risk      0.818967
unhealthy    0.113487
fit          0.067546
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
